# Published speed benchmarks

Four published candidates run on a Ryzen 9 5900HX laptop with integrated AMD Radeon graphics. The saved run uses predecoded 1024 × 768 frames, with two warm-up frames per timed group. The common benchmark uses 720 frame pairs. The published-score checks below use each paper’s scoring convention. [Shared benchmark code](../src/rsc/benchmark.py)

In [1]:
from csv import DictReader
from pathlib import Path
from statistics import mean, median
import subprocess
import sys

from IPython.display import Markdown, display

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "results" / "timings.csv").is_file())

def read_results(filename):
    with (ROOT / "results" / filename).open(encoding="utf-8", newline="") as file:
        return list(DictReader(file))

quality = read_results("quality.csv")
timings = read_results("timings.csv")

def latency(method):
    return mean(float(row["milliseconds"]) for row in timings if row["method"] == method)

def show_table(headers, rows):
    header = "| " + " | ".join(headers) + " |"
    separators = "| " + " | ".join("---" for _ in headers) + " |"
    body = ["| " + " | ".join(map(str, row)) + " |" for row in rows]
    display(Markdown("\n".join([header, separators, *body])))

## Check the published quality scores

JAMNet and Uni-RIFE use their papers’ scoring conventions. DFRSC-2F uses the released weights with a numerically equivalent export. Its scores are close to the paper’s. V-DRSC is a reconstruction from the paper, and its own medians are below the reported medians.

In [2]:
papers = {
    "vdrsc": ("V-DRSC", "https://hal.science/hal-04280534/document", 34.53, .950),
    "jamnet": ("JAMNet", "https://openaccess.thecvf.com/content/CVPR2023/html/Fan_Joint_Appearance_and_Motion_Learning_for_Efficient_Rolling_Shutter_Correction_CVPR_2023_paper.html", 32.93, .941),
    "dfrsc-2f": ("DFRSC-2F", "https://arxiv.org/abs/2404.06350", 33.39, .947),
    "uni-rife": ("Uni-RIFE", "https://doi.org/10.1109/TIP.2024.3504275", 31.60, .927),
}

rows = []
for method, (name, url, reported_psnr, reported_ssim) in papers.items():
    group = [row for row in quality if row["method"] == method]
    summary = median if method == "vdrsc" else mean
    psnr = summary(float(row["paper_psnr_db"]) for row in group)
    ssim = summary(float(row["paper_ssim"]) for row in group)
    rows.append((f"[{name}]({url})", "Median" if method == "vdrsc" else "Mean",
                 len(group), f"{reported_psnr:.2f} / {reported_ssim:.3f}", f"{psnr:.2f} / {ssim:.3f}"))

show_table(("Method", "Aggregation", "Frames", "Published PSNR / SSIM", "Run PSNR / SSIM"), rows)

| Method | Aggregation | Frames | Published PSNR / SSIM | Run PSNR / SSIM |
| --- | --- | --- | --- | --- |
| [V-DRSC](https://hal.science/hal-04280534/document) | Median | 720 | 34.53 / 0.950 | 28.36 / 0.900 |
| [JAMNet](https://openaccess.thecvf.com/content/CVPR2023/html/Fan_Joint_Appearance_and_Motion_Learning_for_Efficient_Rolling_Shutter_Correction_CVPR_2023_paper.html) | Mean | 735 | 32.93 / 0.941 | 32.93 / 0.941 |
| [DFRSC-2F](https://arxiv.org/abs/2404.06350) | Mean | 735 | 33.39 / 0.947 | 33.41 / 0.948 |
| [Uni-RIFE](https://doi.org/10.1109/TIP.2024.3504275) | Mean | 735 | 31.60 / 0.927 | 31.60 / 0.927 |

## Compare speed on the same laptop

Each baseline has 120 recorded correction times. [Timing rows](../results/timings.csv) · [Quality rows](../results/quality.csv)

In [3]:
rows = []
for method in sorted(papers, key=latency):
    backend = next(row["backend"] for row in timings if row["method"] == method)
    rows.append((papers[method][0], backend, f"{latency(method):.2f}"))

show_table(("Method", "Backend", "Mean ms/frame"), rows)


| Method | Backend | Mean ms/frame |
| --- | --- | --- |
| Uni-RIFE | DirectML | 556.99 |
| V-DRSC | CPU | 644.50 |
| JAMNet | DirectML | 658.95 |
| DFRSC-2F | DirectML | 1123.43 |

## Rerun locally

Follow the [setup instructions](../REPRODUCE.md#run-on-windows), then set the run size below.

In [4]:
run = "saved"  # Change to "quick" for three frames, or "full" for all test scenes.
device = "recorded"  # "cpu" runs all methods on the CPU.
data = Path("data/BS-RSC")
if run in ("quick", "full"):
    command = [
        sys.executable, "-u", "-m", "rsc.benchmark", "--suite", "baselines",
        "--" + run, "--device", device, "--data", str(data),
        "--output", "runs/notebook-baselines",
    ]
    with subprocess.Popen(
        command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        encoding="utf-8", errors="replace",
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)
else:
    print("Showing the saved report run.")

Showing the saved report run.
